# Avaliação do STT — Sentinela do Território

**Integrantes:** GEANDRO (RM 562316) · LUCAS VERO (RM 564202) · ROBERTO EDUARDO (RM 564537)

Este notebook mede a qualidade do reconhecimento de voz (STT) do assistente,
usando os 21 áudios de teste gravados pelo grupo. Ele calcula o **WER**
(Word Error Rate — taxa de erro de palavras: quanto menor, melhor) de duas
versões do modelo Whisper (`base` e `small`), no geral, por condição de
ruído e por grupo de voz (análise de viés exigida no barema).

Por que no Colab e não direto no computador: o modelo Whisper precisa ser
baixado da internet (do Hugging Face) na primeira vez que é usado, e esse
download é grande — o Colab já vem com conexão rápida e sem essa restrição.


## 1. Instalar as bibliotecas

`faster-whisper` é a biblioteca de STT (reconhecimento de voz). `jiwer` calcula
o WER (compara a frase que deveria ter sido transcrita com a que o modelo
realmente transcreveu).


In [ ]:
!pip install -q faster-whisper jiwer


## 2. Enviar os áudios e o manifesto

Rode a célula abaixo e, quando aparecer o botão, envie o arquivo
`audios_stt.zip` (contém os 21 áudios e o `manifesto.csv` com a frase, a voz
e a condição de ruído de cada um).


In [ ]:
from google.colab import files
import zipfile
from pathlib import Path

enviado = files.upload()
nome_zip = next(iter(enviado))

PASTA_AUDIOS = Path("/content/audios_stt")
with zipfile.ZipFile(nome_zip) as z:
    z.extractall(PASTA_AUDIOS)

print("Arquivos extraídos:", sorted(p.name for p in PASTA_AUDIOS.iterdir()))


## 3. Ler o manifesto e preparar as funções de avaliação

`normalizar()` deixa a frase de referência e a transcrição no mesmo formato
(minúsculas, sem pontuação) antes de comparar — sem isso, uma diferença de
maiúscula ou de ponto final contaria como erro, o que não é justo.


In [ ]:
import csv
import re

def normalizar(texto: str) -> str:
    texto = re.sub(r"[^\w\s]", " ", texto.lower())
    return re.sub(r"\s+", " ", texto).strip()

with open(PASTA_AUDIOS / "manifesto.csv", encoding="utf-8-sig", newline="") as arquivo:
    linhas = [l for l in csv.DictReader(arquivo) if (l.get("arquivo") or "").strip()]

print(f"{len(linhas)} áudio(s) no manifesto.")


## 4. Transcrever e calcular o WER

Esta célula roda os dois tamanhos de modelo (`base` e `small`), um atrás do
outro, sobre os 21 áudios. A primeira transcrição de cada tamanho demora mais
(baixa o modelo); as próximas são mais rápidas.


In [ ]:
import time
from collections import defaultdict
from faster_whisper import WhisperModel
import jiwer

_MODELOS = {}

def transcrever(caminho_audio, tamanho):
    if tamanho not in _MODELOS:
        _MODELOS[tamanho] = WhisperModel(tamanho, device="cpu", compute_type="int8")
    modelo = _MODELOS[tamanho]
    segmentos, _ = modelo.transcribe(str(caminho_audio), language="pt", vad_filter=True, beam_size=5)
    texto = " ".join(s.text.strip() for s in segmentos).strip()
    if not texto:
        segmentos, _ = modelo.transcribe(str(caminho_audio), language="pt", vad_filter=False, beam_size=5)
        texto = " ".join(s.text.strip() for s in segmentos).strip()
    return texto


def wer_conjunto(pares):
    pares = [(r, h) for r, h in pares if r]
    if not pares:
        return float("nan")
    return jiwer.wer([r for r, _ in pares], [h for _, h in pares])


detalhe, resumo = [], []
for tamanho in ["base", "small"]:
    pares, por_grupo, por_ruido, tempos = [], defaultdict(list), defaultdict(list), []
    for linha in linhas:
        caminho = PASTA_AUDIOS / linha["arquivo"].strip()
        referencia = normalizar(linha.get("referencia", ""))
        inicio = time.perf_counter()
        try:
            hipotese = normalizar(transcrever(caminho, tamanho))
            erro = ""
        except Exception as exc:
            hipotese, erro = "", str(exc)
        duracao = time.perf_counter() - inicio
        wer = jiwer.wer(referencia, hipotese) if referencia else float("nan")
        par = (referencia, hipotese)
        pares.append(par)
        por_grupo[linha.get("grupo", "").strip() or "sem grupo"].append(par)
        por_ruido[linha.get("ruido", "").strip().lower() or "nao"].append(par)
        tempos.append(duracao)
        detalhe.append({"modelo": tamanho, "arquivo": linha["arquivo"], "locutor": linha.get("locutor", ""),
                        "grupo": linha.get("grupo", ""), "ruido": linha.get("ruido", ""),
                        "referencia": referencia, "transcricao": hipotese,
                        "wer": round(wer, 4), "tempo_s": round(duracao, 2), "erro": erro})
        print(f"[{tamanho}] {linha['arquivo']}: WER {wer:.2%} em {duracao:.1f}s — \"{hipotese}\"")
    resumo.append({"modelo": tamanho, "geral": wer_conjunto(pares), "tempo": sum(tempos) / len(tempos),
                   "grupo": {g: wer_conjunto(p) for g, p in por_grupo.items()},
                   "ruido": {r: wer_conjunto(p) for r, p in por_ruido.items()}})

print("\nConcluído.")


## 5. Tabela-resumo e gráficos

Tabela geral (WER e tempo médio por modelo), gráfico de barras comparando os
dois tamanhos de Whisper, e o detalhamento por ruído e por voz (viés do STT).


In [ ]:
import pandas as pd

tabela_geral = pd.DataFrame([{"Modelo": r["modelo"], "WER geral": r["geral"], "Tempo médio (s)": r["tempo"]} for r in resumo])
display(tabela_geral.style.format({"WER geral": "{:.2%}", "Tempo médio (s)": "{:.2f}"}))


In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.bar([r["modelo"] for r in resumo], [r["geral"] for r in resumo], color=["#4C72B0", "#55A868"])
ax1.set_title("WER geral por modelo")
ax1.set_ylabel("WER")

ax2.bar([r["modelo"] for r in resumo], [r["tempo"] for r in resumo], color=["#4C72B0", "#55A868"])
ax2.set_title("Tempo médio de transcrição por áudio")
ax2.set_ylabel("segundos")
plt.tight_layout()
plt.show()


In [ ]:
linhas_ruido = [{"Modelo": r["modelo"], "Ruído": k, "WER": v} for r in resumo for k, v in sorted(r["ruido"].items())]
tabela_ruido = pd.DataFrame(linhas_ruido)
display(tabela_ruido.pivot(index="Ruído", columns="Modelo", values="WER").style.format("{:.2%}"))


In [ ]:
linhas_grupo = [{"Modelo": r["modelo"], "Grupo (voz)": k, "WER": v} for r in resumo for k, v in sorted(r["grupo"].items())]
tabela_grupo = pd.DataFrame(linhas_grupo)
display(tabela_grupo.pivot(index="Grupo (voz)", columns="Modelo", values="WER").style.format("{:.2%}"))


## 6. Salvar e baixar os resultados

Gera `stt_detalhe.csv` (linha a linha) e `stt_resumo.md` (tabelas prontas
para colar no relatório). Depois de baixar, envie os dois arquivos de volta
no chat para eu colocar no relatório e no repositório.


In [ ]:
from google.colab import files

pd.DataFrame(detalhe).to_csv("stt_detalhe.csv", index=False)

md_linhas = ["# Resultado do STT", "", f"Áudios avaliados: {len(linhas)}", "",
             "## WER geral e tempo médio", "", "| Modelo | WER | Tempo médio por áudio (s) |", "| --- | --- | --- |"]
md_linhas += [f"| {r['modelo']} | {r['geral']:.2%} | {r['tempo']:.2f} |" for r in resumo]
md_linhas += ["", "## WER por condição de ruído", "", "| Modelo | Ruído | WER |", "| --- | --- | --- |"]
md_linhas += [f"| {r['modelo']} | {ruido} | {valor:.2%} |" for r in resumo for ruido, valor in sorted(r["ruido"].items())]
md_linhas += ["", "## WER por grupo de locutor (análise de viés)", "", "| Modelo | Grupo | WER |", "| --- | --- | --- |"]
md_linhas += [f"| {r['modelo']} | {grupo} | {valor:.2%} |" for r in resumo for grupo, valor in sorted(r["grupo"].items())]
Path("stt_resumo.md").write_text("\n".join(md_linhas) + "\n", encoding="utf-8")

files.download("stt_detalhe.csv")
files.download("stt_resumo.md")
print("Baixados: stt_detalhe.csv e stt_resumo.md")
